In [ ]:
# Install required packages
!pip install --quiet ipywidgets pymupdf pandas sentence-transformers

print('✓ Packages installed!')

# Searching and filtering and visualizing PDFs

Now let's work with some **data**.

We'll start with some semantic search, like in [the Luanda Leaks investigation](https://qz.com/1786896/ai-for-investigations-sorting-through-the-luanda-leaks)

In [ ]:
from pathlib import Path

import fitz
import pandas as pd
from sentence_transformers import SentenceTransformer

pd.options.display.max_colwidth = 500

In [ ]:
PDF_DIR = Path("data/pdfs")

rows = []

for pdf_file in sorted(PDF_DIR.rglob("*.pdf")):
    with fitz.open(pdf_file) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = " ".join(page.get_text().split())

            if text:
                rows.append({
                    "file": pdf_file.name,
                    "page": page_number,
                    "text": text,
                })

pages = pd.DataFrame(rows)

pages.head()

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(
    pages["text"].tolist(),
    normalize_embeddings=True,
)

pages["embedding"] = list(embeddings)

pages.head()

In [ ]:
query = "police misconduct"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True,
)[0]

pages["score"] = pages["embedding"].apply(
    lambda embedding: embedding @ query_embedding
)

(
    pages
    .sort_values("score", ascending=False)
    .drop('embedding', axis=1)
    .head(10)
)

Great. But what do we do with that???? **Horrible interface!**